# LeetCode 1797 — Design Authentication Manager bằng Python

Link đề: https://leetcode.com/problems/design-authentication-manager/ · Độ khó: **Medium** · Chủ đề: Hash Table, Linked List, Design, Doubly-Linked List

Skeleton LeetCode:

```python
class AuthenticationManager:

    def __init__(self, timeToLive: int):
        ...

    def generate(self, tokenId: str, currentTime: int) -> None:
        ...

    def renew(self, tokenId: str, currentTime: int) -> None:
        ...

    def countUnexpiredTokens(self, currentTime: int) -> int:
        ...


# Your AuthenticationManager object will be instantiated and called as such:
# obj = AuthenticationManager(timeToLive)
# obj.generate(tokenId,currentTime)
# obj.renew(tokenId,currentTime)
# param_3 = obj.countUnexpiredTokens(currentTime)
```

## 1. Mục tiêu học

1. Viết **bộ chạy chuỗi lệnh** mô phỏng cách LeetCode gọi một lớp thiết kế, để test như "Input/Output" của đề.
2. Lưu token bằng `dict` `tokenId → thời điểm hết hạn` và hiểu quy ước **hết hạn tại `t` thì không còn hợp lệ tại `t`** (`expire > currentTime`).
3. Tối ưu `countUnexpiredTokens` bằng **dọn dẹp lười** (lazy deletion) và bằng `OrderedDict` giữ thứ tự hết hạn (khấu hao `O(1)`).
4. Tận dụng giả thiết **thời gian tăng nghiêm ngặt** để chứng minh `OrderedDict` luôn sắp theo thời điểm hết hạn.
5. Đối chiếu mọi cài đặt với một bản **vét cạn phát lại lịch sử** trên chuỗi lệnh ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Hệ thống xác thực dùng **token**. Mỗi phiên, người dùng nhận một token mới, hết hạn sau `timeToLive` giây kể từ `currentTime`. Nếu token được **gia hạn** (renew), thời điểm hết hạn thành `timeToLive` giây kể từ `currentTime` (mới).

Cài lớp `AuthenticationManager`:
- `AuthenticationManager(timeToLive)`: khởi tạo với `timeToLive`.
- `generate(tokenId, currentTime)`: tạo token mới `tokenId` tại thời điểm `currentTime`.
- `renew(tokenId, currentTime)`: gia hạn token **chưa hết hạn** `tokenId`. Nếu không có token chưa hết hạn với id đó, bỏ qua.
- `countUnexpiredTokens(currentTime)`: trả số token chưa hết hạn tại `currentTime`.

**Lưu ý:** nếu một token hết hạn tại thời điểm `t` và một thao tác khác xảy ra đúng lúc `t` (renew hoặc count), việc hết hạn xảy ra **trước**.

**Ví dụ 1:**
```
["AuthenticationManager","renew","generate","countUnexpiredTokens","generate","renew","renew","countUnexpiredTokens"]
[[5],["aaa",1],["aaa",2],[6],["bbb",7],["aaa",8],["bbb",10],[15]]
Output: [null,null,null,1,null,null,null,0]
```
- `renew("aaa", 1)`: chưa có token ⇒ bỏ qua. `generate("aaa", 2)`: hết hạn lúc 7.
- `count(6)`: chỉ `aaa` còn hạn ⇒ 1. `generate("bbb", 7)`: hết hạn lúc 12.
- `renew("aaa", 8)`: `aaa` đã hết hạn lúc 7 ⇒ bỏ qua. `renew("bbb", 10)`: hết hạn mới 15.
- `count(15)`: `bbb` hết hạn đúng lúc 15 ⇒ không tính ⇒ 0.

**Ràng buộc:** `1 ≤ timeToLive, currentTime ≤ 10⁸`; `1 ≤ len(tokenId) ≤ 5`, chữ thường; mọi lời gọi `generate` có `tokenId` **khác nhau**; `currentTime` **tăng nghiêm ngặt** qua các lời gọi; tối đa 2000 lời gọi.

In [1]:
import random
import time
from collections import OrderedDict, defaultdict

def run(cls, ops, args):
    """Bộ chạy kiểu LeetCode: lời gọi đầu là hàm khởi tạo (kết quả None)."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["AuthenticationManager", "renew", "generate", "countUnexpiredTokens", "generate", "renew", "renew", "countUnexpiredTokens"]
ex_args = [[5], ["aaa", 1], ["aaa", 2], [6], ["bbb", 7], ["aaa", 8], ["bbb", 10], [15]]
ex_out = [None, None, None, 1, None, None, None, 0]
print(list(zip(ex_ops, ex_args)), "->", ex_out)

[('AuthenticationManager', [5]), ('renew', ['aaa', 1]), ('generate', ['aaa', 2]), ('countUnexpiredTokens', [6]), ('generate', ['bbb', 7]), ('renew', ['aaa', 8]), ('renew', ['bbb', 10]), ('countUnexpiredTokens', [15])] -> [None, None, None, 1, None, None, None, 0]


## 3. Kiến thức nền

### 3.1 `dict` lưu thời điểm hết hạn

`expire[tokenId] = currentTime + timeToLive`. Token còn hạn tại `t` ⇔ `expire > t` (**nghiêm ngặt**, theo lưu ý "hết hạn xảy ra trước").

In [2]:
ttl = 5; expire = {"aaa": 2 + ttl}
for t in [6, 7, 8]:
    print(f"t={t}: 'aaa' còn hạn? {expire['aaa'] > t}")

t=6: 'aaa' còn hạn? True
t=7: 'aaa' còn hạn? False
t=8: 'aaa' còn hạn? False


### 3.2 `dict.get` với mặc định

`expire.get(tokenId, 0) > t` an toàn khi token không tồn tại (0 luôn `≤ t` vì `t ≥ 1`).

### 3.3 `collections.OrderedDict` — dict nhớ thứ tự, `move_to_end`, `popitem(last=False)`

`move_to_end(k)` đưa khóa `k` về cuối `O(1)`; `popitem(last=False)` lấy phần tử **đầu** `O(1)`; `next(iter(od))` xem khóa đầu.

In [3]:
od = OrderedDict()
od["aaa"] = 7; od["bbb"] = 12
od["aaa"] = 13; od.move_to_end("aaa")
print(list(od.items()), "| đầu:", next(iter(od)))
print("pop đầu:", od.popitem(last=False), "| còn:", list(od.items()))

[('bbb', 12), ('aaa', 13)] | đầu: bbb
pop đầu: ('bbb', 12) | còn: [('aaa', 13)]


### 3.4 Thiết kế lớp và bộ chạy lệnh

Hàm `run` ở trên nhận **tên lớp**, danh sách tên phương thức và danh sách tham số — giống hệt định dạng Input của LeetCode — rồi trả danh sách kết quả (`None` cho các phương thức không trả gì).

## 4. Ý tưởng nền tảng / trực giác

Lập bảng **Ví dụ 1** với `ttl = 5`:

| lệnh | t | trạng thái sau lệnh (id: hết hạn lúc) | kết quả |
|---|---|---|---|
| renew aaa | 1 | {} (bỏ qua) | — |
| generate aaa | 2 | {aaa: 7} | — |
| count | 6 | aaa 7 > 6 ✓ | 1 |
| generate bbb | 7 | {aaa: 7, bbb: 12} | — |
| renew aaa | 8 | aaa 7 ≤ 8 ⇒ hết hạn, bỏ qua | — |
| renew bbb | 10 | {aaa: 7, bbb: 15} | — |
| count | 15 | aaa 7 ≤ 15, bbb 15 ≤ 15 | 0 |

Chỉ cần nhớ **thời điểm hết hạn** của mỗi token. Câu hỏi còn lại: đếm nhanh số token có `expire > t`.

## 5. Cách 1 — Brute Force / baseline: phát lại toàn bộ lịch sử mỗi lần đếm

**Ý tưởng:** chỉ ghi lại các sự kiện `(loại, id, t)`. Khi `count(t)`, phát lại từ đầu mọi sự kiện có thời điểm `≤ t` để dựng lại trạng thái rồi đếm. Đây là cách "không suy nghĩ", bám sát định nghĩa — dùng làm **đáp án chuẩn** để đối chiếu.

In [4]:
class AuthBrute:
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.log = []
    def generate(self, tokenId, currentTime):
        self.log.append(("gen", tokenId, currentTime))
    def renew(self, tokenId, currentTime):
        self.log.append(("renew", tokenId, currentTime))
    def countUnexpiredTokens(self, currentTime):
        expire = {}
        for kind, tid, t in self.log:                       # phát lại lịch sử
            if kind == "gen":
                expire[tid] = t + self.ttl
            elif tid in expire and expire[tid] > t:          # chỉ gia hạn token còn hạn
                expire[tid] = t + self.ttl
        return sum(1 for e in expire.values() if e > currentTime)

got = run(AuthBrute, ex_ops, ex_args); print(got); assert got == ex_out

[None, None, None, 1, None, None, None, 0]


**Độ phức tạp:** mỗi `count` tốn `O(số sự kiện)` ⇒ tổng `O(Q²)` với `Q ≤ 2000` lời gọi — vẫn qua giới hạn nhưng lãng phí: trạng thái có thể **cập nhật dần** thay vì dựng lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (lưu trạng thái trực tiếp):** giữ `expire[tokenId]`; `generate` gán, `renew` chỉ gán khi `expire[tokenId] > t`, `count` đếm `expire > t`.

**Insight 2 (dọn dẹp lười):** token đã hết hạn **không bao giờ** sống lại (không được renew; `generate` dùng id mới). Vì thời gian chỉ tăng, khi `count(t)` thấy token hết hạn thì **xóa** luôn — lần sau không phải xét lại.

**Insight 3 (thứ tự hết hạn = thứ tự cập nhật):** mỗi lần `generate`/`renew` đặt `expire = t + ttl` với `t` tăng nghiêm ngặt ⇒ token được cập nhật **sau** luôn hết hạn **muộn hơn**. Nếu giữ token trong `OrderedDict` theo thứ tự cập nhật gần nhất (renew thì `move_to_end`), các token **đầu** danh sách hết hạn sớm nhất ⇒ khi `count(t)`, chỉ cần **pop từ đầu** chừng nào còn hết hạn. Mỗi token bị pop tối đa một lần ⇒ khấu hao `O(1)`.

**Invariant (Cách 4):** `expire` của các khóa trong `OrderedDict` **tăng nghiêm ngặt** từ đầu tới cuối.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `dict` + đếm bằng cách quét

In [5]:
class AuthDict:
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.expire = {}
    def generate(self, tokenId, currentTime):
        self.expire[tokenId] = currentTime + self.ttl
    def renew(self, tokenId, currentTime):
        if self.expire.get(tokenId, 0) > currentTime:
            self.expire[tokenId] = currentTime + self.ttl
    def countUnexpiredTokens(self, currentTime):
        return sum(1 for e in self.expire.values() if e > currentTime)

print(run(AuthDict, ex_ops, ex_args))

[None, None, None, 1, None, None, None, 0]


**Code nộp LeetCode (Cách 2):** đổi tên lớp thành `AuthenticationManager`. **Độ phức tạp:** `generate`, `renew` `O(1)`; `count` `O(T)` với `T` = số token đã tạo.

### 7.2 Cách 3 — `dict` + dọn dẹp lười khi đếm

In [6]:
class AuthLazy:
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.expire = {}
    def generate(self, tokenId, currentTime):
        self.expire[tokenId] = currentTime + self.ttl
    def renew(self, tokenId, currentTime):
        e = self.expire.get(tokenId)
        if e is not None:
            if e > currentTime:
                self.expire[tokenId] = currentTime + self.ttl
            else:
                del self.expire[tokenId]                 # hết hạn: xóa luôn
    def countUnexpiredTokens(self, currentTime):
        dead = [k for k, e in self.expire.items() if e <= currentTime]
        for k in dead:
            del self.expire[k]
        return len(self.expire)

print(run(AuthLazy, ex_ops, ex_args))

[None, None, None, 1, None, None, None, 0]


**Độ phức tạp Cách 3:** `count` vẫn `O(số token đang lưu)`, nhưng số token đang lưu chỉ gồm token **còn hạn** sau lần đếm trước ⇒ bộ nhớ và thời gian thường nhỏ hơn nhiều.

### 7.3 Cách 4 — `OrderedDict` theo thứ tự hết hạn, khấu hao `O(1)`

In [7]:
class AuthOrdered:
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.od = OrderedDict()
    def _evict(self, t):
        od = self.od
        while od and next(iter(od.values())) <= t:     # phần tử đầu hết hạn sớm nhất
            od.popitem(last=False)
    def generate(self, tokenId, currentTime):
        self._evict(currentTime)
        self.od[tokenId] = currentTime + self.ttl         # id mới => thêm vào cuối
    def renew(self, tokenId, currentTime):
        self._evict(currentTime)
        if tokenId in self.od:
            self.od[tokenId] = currentTime + self.ttl
            self.od.move_to_end(tokenId)
    def countUnexpiredTokens(self, currentTime):
        self._evict(currentTime)
        return len(self.od)

print(run(AuthOrdered, ex_ops, ex_args))

[None, None, None, 1, None, None, None, 0]


**Code nộp LeetCode (Cách 4):** xem mục 9. **Độ phức tạp:** mỗi token được thêm một lần và bị `popitem` tối đa một lần ⇒ tổng `O(Q)` cho `Q` lời gọi — khấu hao `O(1)` mỗi lời gọi.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 — in `OrderedDict` sau mỗi lệnh:

In [8]:
class TraceOrdered(AuthOrdered):
    def _evict(self, t):
        before = list(self.od.items()); super()._evict(t)
        gone = [k for k, _ in before if k not in self.od]
        if gone:
            print(f"    (t={t}) xóa token hết hạn: {gone}")
    def generate(self, tokenId, currentTime):
        super().generate(tokenId, currentTime); print(f"  generate({tokenId!r}, {currentTime}) -> {list(self.od.items())}")
    def renew(self, tokenId, currentTime):
        super().renew(tokenId, currentTime); print(f"  renew({tokenId!r}, {currentTime}) -> {list(self.od.items())}")
    def countUnexpiredTokens(self, currentTime):
        r = super().countUnexpiredTokens(currentTime); print(f"  count({currentTime}) = {r} | {list(self.od.items())}"); return r

assert run(TraceOrdered, ex_ops, ex_args) == ex_out

  renew('aaa', 1) -> []
  generate('aaa', 2) -> [('aaa', 7)]
  count(6) = 1 | [('aaa', 7)]
    (t=7) xóa token hết hạn: ['aaa']
  generate('bbb', 7) -> [('bbb', 12)]
  renew('aaa', 8) -> [('bbb', 12)]
  renew('bbb', 10) -> [('bbb', 15)]
    (t=15) xóa token hết hạn: ['bbb']
  count(15) = 0 | []


| lệnh | OrderedDict sau lệnh (id: hết hạn) | ghi chú |
|---|---|---|
| renew aaa @1 | {} | không có token |
| generate aaa @2 | {aaa: 7} | |
| count @6 | {aaa: 7} | trả 1 |
| generate bbb @7 | {bbb: 12} | trước khi thêm, `aaa` (7 ≤ 7) bị xóa |
| renew aaa @8 | {bbb: 12} | `aaa` không còn ⇒ bỏ qua |
| renew bbb @10 | {bbb: 15} | |
| count @15 | {} | `bbb` 15 ≤ 15 bị xóa ⇒ trả 0 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — `OrderedDict`, khấu hao `O(1)`.

In [9]:
class AuthenticationManager:

    def __init__(self, timeToLive: int):
        self.ttl = timeToLive
        self.od = OrderedDict()                    # tokenId -> thời điểm hết hạn (tăng dần theo thứ tự)

    def _evict(self, currentTime: int) -> None:
        od = self.od
        while od:
            k = next(iter(od))
            if od[k] > currentTime:
                break
            od.popitem(last=False)

    def generate(self, tokenId: str, currentTime: int) -> None:
        self._evict(currentTime)
        self.od[tokenId] = currentTime + self.ttl

    def renew(self, tokenId: str, currentTime: int) -> None:
        self._evict(currentTime)
        if tokenId in self.od:
            self.od[tokenId] = currentTime + self.ttl
            self.od.move_to_end(tokenId)

    def countUnexpiredTokens(self, currentTime: int) -> int:
        self._evict(currentTime)
        return len(self.od)


# Your AuthenticationManager object will be instantiated and called as such:
# obj = AuthenticationManager(timeToLive)
# obj.generate(tokenId,currentTime)
# obj.renew(tokenId,currentTime)
# param_3 = obj.countUnexpiredTokens(currentTime)
got = run(AuthenticationManager, ex_ops, ex_args); print(got); assert got == ex_out

[None, None, None, 1, None, None, None, 0]


## 10. Vì sao thuật toán đúng

**Ngữ nghĩa.** Token `k` còn hạn tại `t` ⇔ `expire[k] > t`, với `expire[k]` = `t_cập_nhật_cuối + ttl` (cập nhật hợp lệ gồm `generate` và các `renew` xảy ra khi token còn hạn). Cách 2 cài đúng nguyên văn điều này.

**Cách 3/4 — xóa token hết hạn là an toàn.** Nếu `expire[k] ≤ t` thì với mọi lời gọi sau (thời điểm `t' > t`), `expire[k] ≤ t'` ⇒ `k` không được tính và `renew(k)` bị bỏ qua; `generate(k)` không xảy ra lại (id duy nhất). Vậy `k` không bao giờ ảnh hưởng kết quả nữa ⇒ xóa được.

**Cách 4 — invariant thứ tự.** Khóa ở cuối `OrderedDict` luôn là khóa vừa cập nhật với `expire = t + ttl`, mà `t` lớn hơn mọi thời điểm cập nhật trước ⇒ `expire` của nó lớn nhất. `move_to_end` giữ tính chất này khi renew. Do đó `expire` tăng nghiêm ngặt từ đầu tới cuối; `_evict` pop từ đầu dừng ở khóa đầu tiên còn hạn ⇒ sau `_evict(t)`, `OrderedDict` chứa **đúng** các token còn hạn. ∎

## 11. Phân tích Time / Space Complexity

Gọi `Q` = số lời gọi (`≤ 2000`), `T` = số token đã tạo.

| Cách | `generate` | `renew` | `count` | Space |
|---|---|---|---|---|
| 1. Phát lại lịch sử | `O(1)` | `O(1)` | `O(Q)` | `O(Q)` |
| 2. Dict + quét | `O(1)` | `O(1)` | `O(T)` | `O(T)` |
| 3. Dict + xóa lười | `O(1)` | `O(1)` | `O(số token đang lưu)` | `O(số token còn hạn)` |
| 4. OrderedDict | `O(1)` khấu hao | `O(1)` khấu hao | `O(1)` khấu hao | `O(số token còn hạn)` |

In [10]:
def random_ops(rng, n_calls, ttl_max=50, id_pool=None):
    ttl = rng.randint(1, ttl_max); ops = ["AuthenticationManager"]; args = [[ttl]]
    t = 0; made = []; counter = 0
    for _ in range(n_calls):
        t += rng.randint(1, max(1, ttl // 2 + 1))
        r = rng.random()
        if r < 0.4:
            counter += 1; tid = f"t{counter}"; made.append(tid)
            ops.append("generate"); args.append([tid, t])
        elif r < 0.75:
            tid = rng.choice(made) if made and rng.random() < 0.85 else "zz"
            ops.append("renew"); args.append([tid, t])
        else:
            ops.append("countUnexpiredTokens"); args.append([t])
    return ops, args

rng = random.Random(1797)
ops, args = random_ops(rng, 20000, ttl_max=10 ** 4)
for cls in (AuthDict, AuthLazy, AuthOrdered, AuthenticationManager):
    t0 = time.perf_counter(); run(cls, ops, args); print(f"{cls.__name__:<22} {(time.perf_counter()-t0)*1000:7.0f} ms cho 20000 lời gọi")

AuthDict                   315 ms cho 20000 lời gọi
AuthLazy                     4 ms cho 20000 lời gọi
AuthOrdered                  6 ms cho 20000 lời gọi
AuthenticationManager        6 ms cho 20000 lời gọi


## 12. Bộ test

Chuỗi lệnh ngẫu nhiên (seed cố định) thỏa ràng buộc: thời gian **tăng nghiêm ngặt**, `generate` với id **mới**, `renew` với id có thể không tồn tại. Đối chiếu mọi cài đặt với `AuthBrute` (phát lại lịch sử). Thêm edge case quanh **ranh giới hết hạn** (`expire == currentTime`).

In [11]:
IMPLS = [AuthDict, AuthLazy, AuthOrdered, AuthenticationManager]
def check(ops, args):
    ref = run(AuthBrute, ops, args)
    for cls in IMPLS:
        assert run(cls, ops, args) == ref, (cls.__name__, ops, args)
    return ref

assert check(ex_ops, ex_args) == ex_out
edges = [
    (["AuthenticationManager", "generate", "countUnexpiredTokens"], [[1], ["a", 1], [2]], [None, None, 0]),      # hết hạn đúng lúc 2
    (["AuthenticationManager", "generate", "countUnexpiredTokens"], [[2], ["a", 1], [2]], [None, None, 1]),
    (["AuthenticationManager", "generate", "renew", "countUnexpiredTokens"], [[3], ["a", 1], ["a", 4], [5]], [None, None, None, 0]),  # renew đúng lúc hết hạn: bỏ qua
    (["AuthenticationManager", "generate", "renew", "countUnexpiredTokens"], [[3], ["a", 1], ["a", 3], [5]], [None, None, None, 1]),
    (["AuthenticationManager", "countUnexpiredTokens"], [[5], [1]], [None, 0]),
]
for ops, args, exp in edges:
    assert check(ops, args) == exp, (ops, args)
rng = random.Random(17970)
for _ in range(400):
    check(*random_ops(rng, rng.randint(1, 60), ttl_max=rng.choice([1, 3, 10, 50])))
for _ in range(5):
    check(*random_ops(rng, 2000, ttl_max=10 ** 3))
print("Ví dụ, edge case ranh giới hết hạn, 400 chuỗi ngắn, 5 chuỗi 2000 lệnh: OK")

Ví dụ, edge case ranh giới hết hạn, 400 chuỗi ngắn, 5 chuỗi 2000 lệnh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` (coi token còn hạn đúng lúc hết hạn)

In [12]:
class WrongGE(AuthDict):
    def renew(self, tokenId, currentTime):
        if self.expire.get(tokenId, 0) >= currentTime:                 # SAI
            self.expire[tokenId] = currentTime + self.ttl
    def countUnexpiredTokens(self, currentTime):
        return sum(1 for e in self.expire.values() if e >= currentTime)   # SAI

print("sai :", run(WrongGE, ex_ops, ex_args), "| đúng:", ex_out)

sai : [None, None, None, 1, None, None, None, 1] | đúng: [None, None, None, 1, None, None, None, 0]


**Vì sao sai:** đề nói việc hết hạn xảy ra **trước** thao tác cùng thời điểm ⇒ token hết hạn lúc 15 không được đếm ở `count(15)`. **Sửa:** `expire > currentTime`.

### Lỗi 2 — `renew` cả token đã hết hạn (hoặc tạo mới nếu chưa có)

In [13]:
class WrongRenewAll(AuthDict):
    def renew(self, tokenId, currentTime):
        self.expire[tokenId] = currentTime + self.ttl                   # SAI: không kiểm tra còn hạn / tồn tại

print("Ví dụ 1 — sai :", run(WrongRenewAll, ex_ops, ex_args), "| đúng:", ex_out, "(tình cờ trùng)")
ops = ["AuthenticationManager", "renew", "generate", "renew", "countUnexpiredTokens"]
args = [[3], ["z", 1], ["a", 2], ["a", 6], [7]]          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("Tự dựng — sai :", run(WrongRenewAll, ops, args), "| đúng:", run(AuthenticationManager, ops, args), "| brute:", run(AuthBrute, ops, args))

Ví dụ 1 — sai : [None, None, None, 1, None, None, None, 0] | đúng: [None, None, None, 1, None, None, None, 0] (tình cờ trùng)
Tự dựng — sai : [None, None, None, None, 1] | đúng: [None, None, None, None, 0] | brute: [None, None, None, None, 0]


**Vì sao sai:** ở Ví dụ 1 lỗi bị "che" vì token hồi sinh lại hết hạn trước lần đếm cuối. Ở ví dụ tự dựng: `renew("z", 1)` **tạo** ra token `z` không tồn tại (hết hạn lúc 4), còn `renew("a", 6)` **hồi sinh** token `a` đã hết hạn lúc 5 thành hết hạn lúc 9 ⇒ `count(7)` ra 1 thay vì 0. **Sửa:** chỉ gia hạn khi token tồn tại **và** `expire > currentTime`.

### Lỗi 3 — Cách 4 nhưng quên `move_to_end` khi renew

In [14]:
class WrongNoMove(AuthOrdered):
    def renew(self, tokenId, currentTime):
        self._evict(currentTime)
        if tokenId in self.od:
            self.od[tokenId] = currentTime + self.ttl                  # SAI: thứ tự không còn tăng dần

ops = ["AuthenticationManager", "generate", "generate", "renew", "countUnexpiredTokens"]
args = [[5], ["a", 1], ["b", 2], ["a", 4], [8]]                        # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongNoMove, ops, args), "| đúng:", run(AuthenticationManager, ops, args), "| brute:", run(AuthBrute, ops, args))

sai : [None, None, None, None, 2] | đúng: [None, None, None, None, 1] | brute: [None, None, None, None, 1]


**Vì sao sai:** sau `renew("a", 4)`, `a` hết hạn lúc 9 nhưng vẫn đứng **trước** `b` (hết hạn lúc 7). Ở `count(8)`, `_evict` gặp `a` (9 > 8) ở đầu và **dừng**, không xóa `b` đã hết hạn ⇒ đếm thừa. **Sửa:** `move_to_end` để giữ thứ tự hết hạn.

### Lỗi 4 — Lưu thời điểm **tạo** thay vì thời điểm **hết hạn**, rồi so sai

In [15]:
class WrongStoreStart(AuthDict):
    def generate(self, tokenId, currentTime):
        self.expire[tokenId] = currentTime                            # lưu thời điểm bắt đầu
    def renew(self, tokenId, currentTime):
        s = self.expire.get(tokenId)
        if s is not None and currentTime - s <= self.ttl:             # SAI: phải là < ttl
            self.expire[tokenId] = currentTime
    def countUnexpiredTokens(self, currentTime):
        return sum(1 for s in self.expire.values() if currentTime - s <= self.ttl)   # SAI

print("sai :", run(WrongStoreStart, ex_ops, ex_args), "| đúng:", ex_out)

sai : [None, None, None, 1, None, None, None, 1] | đúng: [None, None, None, 1, None, None, None, 0]


**Vì sao sai:** lưu thời điểm bắt đầu cũng được, nhưng điều kiện "còn hạn" là `currentTime − start < ttl` (nghiêm ngặt). Dễ nhầm hơn so với lưu thẳng thời điểm hết hạn. **Sửa:** lưu `currentTime + ttl` và so `> currentTime`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài design có **thời gian tăng dần** ⇒ thứ tự cập nhật = thứ tự hết hạn ⇒ hàng đợi / `OrderedDict` / danh sách liên kết đôi + hash map.
- Đối tượng hết hạn không bao giờ sống lại ⇒ **xóa lười** an toàn.
- Luôn đọc kỹ quy ước ranh giới (`>` hay `≥`).
- Viết **bộ chạy lệnh** + bản vét cạn để test chuỗi lệnh ngẫu nhiên.

## 15. Biến thể của bài

1. **Thời gian không tăng dần** (lời gọi đến lộn xộn): không dùng được thứ tự cập nhật; cần heap theo `expire` với **xóa lười theo phiên bản** (Bài tập 5).
2. **146 LRU Cache**: hash map + danh sách liên kết đôi / `OrderedDict` — cùng kỹ thuật `move_to_end`.
3. **359 Logger Rate Limiter**, **362 Design Hit Counter**: đếm sự kiện trong cửa sổ thời gian (hàng đợi).
4. **Truy vấn "token sắp hết hạn sớm nhất"**: phần tử đầu `OrderedDict`.
5. **Mỗi token có `ttl` riêng**: thứ tự cập nhật không còn là thứ tự hết hạn ⇒ heap.

## 16. So sánh các cách làm

| Cách | Ý tưởng | `count` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Phát lại lịch sử | Dựng lại trạng thái mỗi lần | `O(Q)` | `O(Q)` | Đối chiếu |
| 2. Dict + quét | Đếm `expire > t` | `O(T)` | `O(T)` | Đủ qua đề (`Q ≤ 2000`) |
| 3. Dict + xóa lười | Xóa token hết hạn khi gặp | `O(token đang lưu)` | nhỏ | Tiết kiệm bộ nhớ |
| 4. OrderedDict | Pop đầu khi hết hạn | `O(1)` khấu hao | nhỏ | **Bản nộp**, tối ưu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `ttl = 3`, lệnh: `generate a@1, generate b@2, count@3, count@4, renew b@4, count@6`.

**Bài 2 (tính tay):** Với `ttl = 10`: `generate x@1, renew x@5, renew x@15, count@16, generate y@20, count@25` — trạng thái sau mỗi lệnh và các kết quả.

**Bài 3 (điền TODO):** Hoàn thiện `is_alive(expire, tokenId, t)`.

**Bài 4 (tự viết lại):** Viết lớp `AuthLinked` dùng **danh sách liên kết đôi tự cài** + `dict` (không dùng `OrderedDict`) với độ phức tạp như Cách 4.

**Bài 5 (mini challenge):** Viết `AuthHeap` khi `currentTime` **không** đảm bảo tăng giữa các lời gọi `generate/renew` (nhưng mọi truy vấn `count` vẫn tăng dần) — dùng heap `(expire, tokenId)` với kiểm tra phiên bản.

In [16]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_alive_todo(expire, tokenId, t):
    # TODO: trả True nếu tokenId có trong expire và expire[tokenId] > t
    pass

In [17]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class AuthLinked:
    def __init__(self, timeToLive):
        # TODO: nút giả head/tail, dict id -> nút
        pass

class AuthHeap:
    def __init__(self, timeToLive):
        # TODO: dict id -> expire, heap (expire, id), tập dead; khi pop, chỉ đánh dấu dead nếu expire == dict[id]
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `a` hết hạn 4, `b` hết hạn 5. Mức 2: `count@3` = 2; `count@4` = 1 (`a` hết hạn đúng lúc 4). Mức 3: `renew b@4` → `b` hết hạn 7; `count@6` = 1.

**Bài 2.** Mức 1: `x` hết hạn 11; renew@5 → 15. Mức 2: renew@15 bị bỏ qua (15 ≤ 15). Mức 3: `count@16` = 0; `generate y@20` → 30; `count@25` = 1.

**Bài 3.** Mức 1: `expire.get(tokenId, 0)`. Mức 2: so `> t`. Mức 3: trả bool.

**Bài 4.** Mức 1: lớp `Node(id, exp, prev, next)`. Mức 2: `generate` nối vào trước `tail`; `renew` gỡ nút rồi nối lại cuối. Mức 3: `_evict` gỡ từ `head.next` khi hết hạn.

**Bài 5.** Mức 1: heap `(expire, id)` + `dict` thời điểm hết hạn hiện tại. Mức 2: `count(t)` (tăng dần): pop khi `heap[0][0] ≤ t`; chỉ khi giá trị pop ra **bằng** `dict[id]` (phiên bản hiện tại) mới đánh dấu `id` là "đã chết". Mức 3: **không** xóa khỏi `dict` — một `renew` đến sau với thời gian nhỏ hơn vẫn có thể gia hạn (ngữ nghĩa "áp dụng theo thứ tự gọi"); khi đó bỏ dấu "đã chết" và đẩy phiên bản mới vào heap.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
ops1 = ["AuthenticationManager", "generate", "generate", "countUnexpiredTokens", "countUnexpiredTokens", "renew", "countUnexpiredTokens"]
args1 = [[3], ["a", 1], ["b", 2], [3], [4], ["b", 4], [6]]
b1 = check(ops1, args1); print("Bài 1:", b1); assert b1 == [None, None, None, 2, 1, None, 1]
ops2 = ["AuthenticationManager", "generate", "renew", "renew", "countUnexpiredTokens", "generate", "countUnexpiredTokens"]
args2 = [[10], ["x", 1], ["x", 5], ["x", 15], [16], ["y", 20], [25]]
b2 = check(ops2, args2); print("Bài 2:", b2); assert b2 == [None, None, None, None, 0, None, 1]
def is_alive_todo(expire, tokenId, t):
    return expire.get(tokenId, 0) > t
assert is_alive_todo({"a": 7}, "a", 6) and not is_alive_todo({"a": 7}, "a", 7) and not is_alive_todo({}, "a", 1)
print("Bài 3: OK")

Bài 1: [None, None, None, 2, 1, None, 1]
Bài 2: [None, None, None, None, 0, None, 1]
Bài 3: OK


In [19]:
# Bài 4
class _Node:
    __slots__ = ("id", "exp", "prev", "next")
    def __init__(self, id=None, exp=0):
        self.id, self.exp, self.prev, self.next = id, exp, None, None

class AuthLinked:
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.map = {}
        self.head, self.tail = _Node(), _Node(); self.head.next, self.tail.prev = self.tail, self.head
    def _unlink(self, nd):
        nd.prev.next, nd.next.prev = nd.next, nd.prev
    def _append(self, nd):
        nd.prev, nd.next = self.tail.prev, self.tail; self.tail.prev.next = nd; self.tail.prev = nd
    def _evict(self, t):
        while self.head.next is not self.tail and self.head.next.exp <= t:
            nd = self.head.next; self._unlink(nd); del self.map[nd.id]
    def generate(self, tokenId, currentTime):
        self._evict(currentTime); nd = _Node(tokenId, currentTime + self.ttl); self.map[tokenId] = nd; self._append(nd)
    def renew(self, tokenId, currentTime):
        self._evict(currentTime)
        nd = self.map.get(tokenId)
        if nd:
            nd.exp = currentTime + self.ttl; self._unlink(nd); self._append(nd)
    def countUnexpiredTokens(self, currentTime):
        self._evict(currentTime); return len(self.map)

rng = random.Random(4)
for _ in range(300):
    ops, args = random_ops(rng, rng.randint(1, 50), ttl_max=rng.choice([2, 10, 40]))
    assert run(AuthLinked, ops, args) == run(AuthBrute, ops, args)
print("Bài 4: OK |", run(AuthLinked, ex_ops, ex_args))

Bài 4: OK | [None, None, None, 1, None, None, None, 0]


In [20]:
# Bài 5
import heapq
class AuthHeap:
    """exp giữ thời điểm hết hạn hiện tại của MỌI token (không xóa, vì renew với thời gian nhỏ hơn có thể hồi sinh);
    'dead' = tập token có phiên bản hiện tại đã hết hạn tại lần count gần nhất."""
    def __init__(self, timeToLive):
        self.ttl = timeToLive; self.exp = {}; self.h = []; self.dead = set()
    def generate(self, tokenId, currentTime):
        e = currentTime + self.ttl; self.exp[tokenId] = e; heapq.heappush(self.h, (e, tokenId))
    def renew(self, tokenId, currentTime):
        if self.exp.get(tokenId, 0) > currentTime:
            e = currentTime + self.ttl; self.exp[tokenId] = e; heapq.heappush(self.h, (e, tokenId))
            self.dead.discard(tokenId)               # phiên bản mới: chưa biết đã hết hạn hay chưa
    def countUnexpiredTokens(self, currentTime):
        while self.h and self.h[0][0] <= currentTime:
            e, tid = heapq.heappop(self.h)
            if self.exp.get(tid) == e:             # chỉ phiên bản HIỆN TẠI mới quyết định
                self.dead.add(tid)
        return len(self.exp) - len(self.dead)

class AuthBruteAny(AuthBrute):
    """Vét cạn cho thời gian không tăng: áp dụng sự kiện theo thứ tự gọi."""
    pass

def random_ops_unordered(rng, n_calls):
    ttl = rng.randint(1, 20); ops = ["AuthenticationManager"]; args = [[ttl]]; made = []; c = 0; tq = 0
    for _ in range(n_calls):
        r = rng.random()
        if r < 0.4:
            c += 1; made.append(f"t{c}"); ops.append("generate"); args.append([f"t{c}", rng.randint(1, 60)])
        elif r < 0.7 and made:
            ops.append("renew"); args.append([rng.choice(made), rng.randint(1, 60)])
        else:
            tq += rng.randint(0, 5); ops.append("countUnexpiredTokens"); args.append([tq])
    return ops, args

rng = random.Random(5)
for _ in range(300):
    ops, args = random_ops_unordered(rng, rng.randint(1, 30))
    assert run(AuthHeap, ops, args) == run(AuthBruteAny, ops, args), (ops, args)
print("Bài 5: OK (heap + kiểm tra phiên bản khớp vét cạn khi thời gian generate/renew không tăng)")

Bài 5: OK (heap + kiểm tra phiên bản khớp vét cạn khi thời gian generate/renew không tăng)


Ghi chú Bài 5: `AuthBruteAny` phát lại sự kiện **theo thứ tự gọi**, đúng với ngữ nghĩa "mỗi lệnh tác động lên trạng thái hiện tại" — một mô hình tự đặt ra cho biến thể (không có trong đề gốc; nên đối chiếu lại nếu gặp bài thật).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "map + thứ tự hết hạn" (thời gian tăng dần):**

```python
od = OrderedDict()                       # id -> expire, expire tăng dần
def evict(t):
    while od and next(iter(od.values())) <= t: od.popitem(last=False)
generate: evict(t); od[id] = t + ttl
renew:    evict(t); if id in od: od[id] = t + ttl; od.move_to_end(id)
count:    evict(t); return len(od)
```

**Checklist:**
- [ ] Còn hạn ⇔ `expire > currentTime` (nghiêm ngặt)?
- [ ] `renew` chỉ khi token **tồn tại và còn hạn**; không tạo mới?
- [ ] Dùng `OrderedDict` thì **`move_to_end`** khi renew?
- [ ] Xóa lười chỉ an toàn khi thời gian **tăng dần** và id `generate` là duy nhất?
- [ ] Đã test ranh giới `expire == currentTime` cho cả `renew` và `count`, chuỗi lệnh ngẫu nhiên đối chiếu vét cạn?